# MovieLens ratings: Bronze Catalog table to silver

This AWS Glue notebook reads `movielens.ratings` as a `DynamicFrame`, converts it to a Spark `DataFrame` for cleansing, converts it back to a `DynamicFrame`, and writes partitioned Parquet while registering `movielens.ratings_silver`.

## Before running

Run in an AWS Glue notebook/interactive session. The execution role needs read access to the source, write access to the silver S3 prefix, and Glue Catalog read/update permissions. Replace `SILVER_ROOT`; use an empty dedicated prefix on the first run.

In [ ]:
from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql import types as T

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

DATABASE = "movielens"
SOURCE_TABLE = "ratings"
TARGET_TABLE = "ratings_silver"
SILVER_ROOT = "s3://YOUR-BUCKET/silver/movielens".rstrip("/")
TARGET_PATH = f"{SILVER_ROOT}/{TARGET_TABLE}/"

assert "YOUR-BUCKET" not in SILVER_ROOT, "Set SILVER_ROOT before running"
print(f"Source: {DATABASE}.{SOURCE_TABLE}")
print(f"Target: {DATABASE}.{TARGET_TABLE} -> {TARGET_PATH}")

## Read from the Glue Data Catalog as a DynamicFrame

In [ ]:
ratings_raw_dyf = glueContext.create_dynamic_frame.from_catalog(
    database=DATABASE,
    table_name=SOURCE_TABLE,
    transformation_ctx="ratings_raw_dyf",
)
print(f"Raw rows: {ratings_raw_dyf.count():,}")
ratings_raw_dyf.printSchema()

## DynamicFrame → DataFrame

The epoch timestamp is converted to a real UTC timestamp. The session timezone is fixed to UTC so the derived event date and year are stable.

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
ratings_raw_df = ratings_raw_dyf.toDF()

def source_column(df, expected):
    matches = [name for name in df.columns if name.lower() == expected.lower()]
    if len(matches) != 1:
        raise ValueError(f"Expected one column named {expected!r}; found {matches}. Available: {df.columns}")
    return F.col(f"`{matches[0]}`")

user_id = source_column(ratings_raw_df, "userId")
movie_id = source_column(ratings_raw_df, "movieId")
rating = source_column(ratings_raw_df, "rating")
rating_epoch = source_column(ratings_raw_df, "timestamp")

In [ ]:
ratings_silver_df = (
    ratings_raw_df
    .select(
        user_id.cast(T.LongType()).alias("user_id"),
        movie_id.cast(T.LongType()).alias("movie_id"),
        rating.cast(T.DoubleType()).alias("rating"),
        rating_epoch.cast(T.LongType()).alias("rating_epoch"),
    )
    .withColumn("rated_at_utc", F.to_timestamp(F.from_unixtime("rating_epoch")))
    .withColumn("event_date", F.to_date("rated_at_utc"))
    .withColumn("event_year", F.year("rated_at_utc"))
    .withColumn("ingested_at_utc", F.current_timestamp())
    .filter(
        F.col("user_id").isNotNull()
        & F.col("movie_id").isNotNull()
        & F.col("rating").between(0.5, 5.0)
        & F.col("rated_at_utc").isNotNull()
    )
    .dropDuplicates(["user_id", "movie_id", "rating_epoch"])
    .select("user_id", "movie_id", "rating", "rating_epoch", "rated_at_utc", "event_date", "ingested_at_utc", "event_year")
)

ratings_silver_df.printSchema()
ratings_silver_df.show(10, truncate=False)

## DataFrame → DynamicFrame and validation

In [ ]:
ratings_silver_dyf = DynamicFrame.fromDF(ratings_silver_df, glueContext, "ratings_silver_dyf")
ratings_silver_dyf.printSchema()

raw_count = ratings_raw_dyf.count()
silver_count = ratings_silver_dyf.count()
invalid_count = ratings_silver_df.filter(
    F.col("user_id").isNull() | F.col("movie_id").isNull() | ~F.col("rating").between(0.5, 5.0)
).count()
duplicate_count = ratings_silver_df.groupBy("user_id", "movie_id", "rating_epoch").count().filter(F.col("count") > 1).count()
print({"raw_count": raw_count, "silver_count": silver_count, "invalid_count": invalid_count, "duplicate_ratings": duplicate_count})
assert invalid_count == 0
assert duplicate_count == 0

## Write partitioned Parquet and update the Catalog

The Glue sink writes data files and creates or updates `movielens.ratings_silver`, partitioned by `event_year`. Glue sinks append files; for repeatable full refreshes, clear only this dedicated target prefix through an approved lifecycle/job step before rerunning, or adopt a versioned/incremental design.

In [ ]:
sink = glueContext.getSink(
    connection_type="s3",
    path=TARGET_PATH,
    enableUpdateCatalog=True,
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["event_year"],
    transformation_ctx="ratings_silver_sink",
)
sink.setCatalogInfo(catalogDatabase=DATABASE, catalogTableName=TARGET_TABLE)
sink.setFormat("glueparquet", compression="snappy")
sink.writeFrame(ratings_silver_dyf)
print(f"Published {DATABASE}.{TARGET_TABLE} at {TARGET_PATH}")

In [ ]:
published_df = glueContext.create_dynamic_frame.from_catalog(
    database=DATABASE, table_name=TARGET_TABLE, transformation_ctx="ratings_published_check"
).toDF()
published_df.printSchema()
published_df.show(10, truncate=False)